# Duplicate Position Analysis

This notebook analyzes files in `outputs/src_outputs/new_vcf/full_set` to find cases where `(POS, new_ref, ALT_new)` combinations appear more than once within a file.

Files listed in `skipped_files.csv` are excluded from analysis.

In [1]:
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Define paths
data_dir = Path('../outputs/src_outputs/new_vcf/full_set')
skipped_file = data_dir / 'skipped_files.csv'

print(f"Data directory: {data_dir}")
print(f"Directory exists: {data_dir.exists()}")

Data directory: ../outputs/src_outputs/new_vcf/full_set
Directory exists: True


In [3]:
# Load skipped file IDs
skipped_df = pd.read_csv(skipped_file)
skipped_ids = set(skipped_df['Run'].tolist())

print(f"Number of IDs to skip: {len(skipped_ids)}")
print(f"First 5 skipped IDs: {list(skipped_ids)[:5]}")

Number of IDs to skip: 201
First 5 skipped IDs: ['ERR3242590', 'ERR3988818', 'ERR3242540', 'ERR3243005', 'ERR3242553']


In [4]:
# Get all CSV files (excluding skipped_files.csv itself)
all_csv_files = [f for f in data_dir.glob('new_vcf_*.csv')]

# Filter out files whose ID is in skipped_ids
def extract_id_from_filename(filepath):
    """Extract the ID from filename like 'new_vcf_ERR3989083.csv' -> 'ERR3989083'"""
    name = filepath.stem  # 'new_vcf_ERR3989083'
    return name.replace('new_vcf_', '')

files_to_analyze = []
skipped_count = 0

for f in all_csv_files:
    file_id = extract_id_from_filename(f)
    if file_id not in skipped_ids:
        files_to_analyze.append(f)
    else:
        skipped_count += 1

print(f"Total CSV files found: {len(all_csv_files)}")
print(f"Files skipped (in skipped_files.csv): {skipped_count}")
print(f"Files to analyze: {len(files_to_analyze)}")

Total CSV files found: 3189
Files skipped (in skipped_files.csv): 201
Files to analyze: 2988


In [5]:
def count_duplicate_positions(filepath):
    """
    Count cases where (POS, new_ref, ALT_new) appear more than once in a file.
    
    Returns:
        - num_duplicate_cases: Number of unique (POS, new_ref, ALT_new) combos that appear >1 time
        - total_duplicate_rows: Total number of rows involved in duplicates
        - duplicate_details: List of (POS, new_ref, ALT_new, count) for each duplicate case
    """
    try:
        df = pd.read_csv(filepath)
        
        # Standardize column names (some files may have different casing)
        df.columns = df.columns.str.strip()
        
        # Check if required columns exist
        required_cols = ['POS', 'new_ref', 'ALT_new']
        if not all(col in df.columns for col in required_cols):
            return None, None, None, f"Missing columns. Found: {list(df.columns)}"
        
        # Group by the three columns and count occurrences
        grouped = df.groupby(['POS', 'new_ref', 'ALT_new']).size().reset_index(name='count')
        
        # Filter to only those with count > 1 (duplicates)
        duplicates = grouped[grouped['count'] > 1]
        
        num_duplicate_cases = len(duplicates)
        total_duplicate_rows = duplicates['count'].sum() if num_duplicate_cases > 0 else 0
        
        duplicate_details = []
        for _, row in duplicates.iterrows():
            duplicate_details.append({
                'POS': row['POS'],
                'new_ref': row['new_ref'],
                'ALT_new': row['ALT_new'],
                'count': row['count']
            })
        
        return num_duplicate_cases, total_duplicate_rows, duplicate_details, None
        
    except Exception as e:
        return None, None, None, str(e)

In [6]:
# Analyze all files
results = []
all_duplicate_details = []  # To track which positions are duplicated across all files
error_files = []

print("Analyzing files...")
for i, filepath in enumerate(files_to_analyze):
    file_id = extract_id_from_filename(filepath)
    num_cases, total_rows, details, error = count_duplicate_positions(filepath)
    
    if error:
        error_files.append({'file_id': file_id, 'error': error})
    else:
        results.append({
            'file_id': file_id,
            'filepath': str(filepath),
            'num_duplicate_cases': num_cases,
            'total_duplicate_rows': total_rows
        })
        
        # Track individual duplicate details
        for d in details:
            d['file_id'] = file_id
            all_duplicate_details.append(d)
    
    # Progress indicator
    if (i + 1) % 500 == 0:
        print(f"  Processed {i + 1}/{len(files_to_analyze)} files...")

print(f"\nDone! Analyzed {len(results)} files successfully.")
if error_files:
    print(f"Files with errors: {len(error_files)}")

Analyzing files...
  Processed 500/2988 files...
  Processed 1000/2988 files...
  Processed 1500/2988 files...
  Processed 2000/2988 files...
  Processed 2500/2988 files...

Done! Analyzed 2988 files successfully.


In [7]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df.head(10)

,file_id,filepath,num_duplicate_cases,total_duplicate_rows
0,ERR3243102,../outputs/src_outputs/new_vcf/full_set/new_vc...,1,2
1,ERR3241715,../outputs/src_outputs/new_vcf/full_set/new_vc...,1,2
2,ERR3239876,../outputs/src_outputs/new_vcf/full_set/new_vc...,0,0
3,ERR3239862,../outputs/src_outputs/new_vcf/full_set/new_vc...,0,0
4,ERR3241701,../outputs/src_outputs/new_vcf/full_set/new_vc...,0,0
5,ERR3242208,../outputs/src_outputs/new_vcf/full_set/new_vc...,0,0
6,ERR3243116,../outputs/src_outputs/new_vcf/full_set/new_vc...,1,2
7,ERR3241729,../outputs/src_outputs/new_vcf/full_set/new_vc...,0,0
8,ERR3242220,../outputs/src_outputs/new_vcf/full_set/new_vc...,0,0
9,ERR3242234,../outputs/src_outputs/new_vcf/full_set/new_vc...,0,0


## Summary Statistics

In [8]:
# Overall summary
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"\nTotal files analyzed: {len(results_df)}")
print(f"\nFiles WITH duplicate (POS, new_ref, ALT_new) cases: {(results_df['num_duplicate_cases'] > 0).sum()}")
print(f"Files WITHOUT any duplicates: {(results_df['num_duplicate_cases'] == 0).sum()}")
print(f"\nTotal duplicate cases across all files: {results_df['num_duplicate_cases'].sum()}")
print(f"Total duplicate rows across all files: {results_df['total_duplicate_rows'].sum()}")
print(f"\nMean duplicate cases per file: {results_df['num_duplicate_cases'].mean():.2f}")
print(f"Max duplicate cases in a single file: {results_df['num_duplicate_cases'].max()}")
print(f"Median duplicate cases per file: {results_df['num_duplicate_cases'].median():.2f}")

SUMMARY STATISTICS

Total files analyzed: 2988

Files WITH duplicate (POS, new_ref, ALT_new) cases: 632
Files WITHOUT any duplicates: 2356

Total duplicate cases across all files: 704
Total duplicate rows across all files: 1408

Mean duplicate cases per file: 0.24
Max duplicate cases in a single file: 4
Median duplicate cases per file: 0.00


In [10]:
# Files with the most duplicates
print("\nTop 20 files with most duplicate cases:")
top_files = results_df.nlargest(20, 'num_duplicate_cases')[['file_id', 'num_duplicate_cases', 'total_duplicate_rows']]
print(top_files.to_string(index=False))


Top 20 files with most duplicate cases:
   file_id  num_duplicate_cases  total_duplicate_rows
ERR3242381                    4                     8
ERR3239280                    3                     6
ERR3239952                    3                     6
ERR3239925                    3                     6
ERR3239926                    3                     6
ERR3239490                    2                     4
ERR3242396                    2                     4
ERR3243063                    2                     4
ERR3243077                    2                     4
ERR3242143                    2                     4
ERR3243049                    2                     4
ERR3239928                    2                     4
ERR3239520                    2                     4
ERR3239454                    2                     4
ERR3243139                    2                     4
ERR3243067                    2                     4
ERR3242386                    2          

## Detailed Analysis of Duplicate Positions

In [15]:
# Create DataFrame of all duplicate details
if all_duplicate_details:
    duplicates_df = pd.DataFrame(all_duplicate_details)
    print(f"Total individual duplicate cases: {len(duplicates_df)}")
    print("\nSample of duplicate cases:")
    display(duplicates_df)
else:
    print("No duplicates found in any file!")
    duplicates_df = pd.DataFrame()

Total individual duplicate cases: 704

Sample of duplicate cases:


,POS,new_ref,ALT_new,count,file_id
0,10882,C,G,2,ERR3243102
1,10882,C,G,2,ERR3241715
2,10882,C,G,2,ERR3243116
3,10882,C,G,2,ERR3239679
4,10882,C,G,2,ERR3239889
...,...,...,...,...,...
699,7719,T,C,2,ERR3242761
700,7719,T,C,2,ERR3239890
701,7719,T,C,2,ERR3242239
702,10882,C,G,2,ERR3242239


In [12]:
# Which positions are most commonly duplicated across files?
if len(duplicates_df) > 0:
    pos_counts = duplicates_df.groupby(['POS', 'new_ref', 'ALT_new']).agg({
        'file_id': 'count',
        'count': 'sum'
    }).rename(columns={'file_id': 'num_files_with_dup', 'count': 'total_occurrences'}).reset_index()
    
    pos_counts_sorted = pos_counts.sort_values('num_files_with_dup', ascending=False)
    
    print("Most commonly duplicated (POS, new_ref, ALT_new) combinations across files:")
    print("(Shows how many different files have this position duplicated)")
    print(pos_counts_sorted.head(30).to_string(index=False))

Most commonly duplicated (POS, new_ref, ALT_new) combinations across files:
(Shows how many different files have this position duplicated)
  POS new_ref ALT_new  num_files_with_dup  total_occurrences
10882       C       G                 402                804
 7719       T       C                 153                306
 7994       G       A                  71                142
12047       G       C                  26                 52
10093       C       G                  20                 40
  673       C       G                  11                 22
 7661       G       C                   3                  6
 3090       C       G                   3                  6
 3002       G       T                   2                  4
 2392       C       G                   2                  4
  674       G       C                   2                  4
  524       G       C                   2                  4
 3006       C       T                   1                  2
  560  

In [13]:
# Error files if any
if error_files:
    print(f"\nFiles with errors ({len(error_files)}):")
    error_df = pd.DataFrame(error_files)
    display(error_df)

## Export Results

In [16]:
# Save results to CSV
output_dir = Path('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/scripts_outputs')

# File-level summary
results_df.to_csv(output_dir / 'duplicate_analysis_by_file.csv', index=False)
print(f"Saved file-level results to: duplicate_analysis_by_file.csv")

# Detailed duplicate cases
if len(duplicates_df) > 0:
    duplicates_df.to_csv(output_dir / 'duplicate_cases_detailed.csv', index=False)
    print(f"Saved detailed duplicate cases to: duplicate_cases_detailed.csv")
    
    pos_counts_sorted.to_csv(output_dir / 'duplicate_positions_summary.csv', index=False)
    print(f"Saved position summary to: duplicate_positions_summary.csv")

Saved file-level results to: duplicate_analysis_by_file.csv
Saved detailed duplicate cases to: duplicate_cases_detailed.csv
Saved position summary to: duplicate_positions_summary.csv
